# 環境建置

In [1]:
!pip install -q mediapipe==0.10.21
!pip install grpcio-status==1.44.0

In [2]:
import cv2
import mediapipe as mp
import numpy as np
import time
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript, Image, HTML, clear_output
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
from PIL import Image as PILImage

# 實作

## take_photo

In [8]:
# 定義JavaScript函數來獲取單張照片
# 這個函數會建立一個介面, 允許使用者點擊按鈕拍照
def take_photo(filename='photo.jpg', quality=0.8):
  """
  使用JavaScript取得使用者的單張照片

  參數:
      filename (str): 儲存照片的檔案名稱, 預設為'photo.jpg'
      quality (float): 照片品質, 介於0到1之間, 預設為0.8

  回傳:
      str: 儲存照片的檔案路徑
  """
  # 定義JavaScript程式碼, 用於存取使用者的攝影機並拍照
  js = Javascript('''
    async function takePhoto(quality) {
      // 建立包含按鈕和視訊元素的div
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '開始攝影'; // 設定按鈕文字
      div.appendChild(capture);

      // 建立視訊元素
      const video = document.createElement('video');
      video.style.display = 'block';
      // 請求使用者的攝影機權限
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      // 將元素添加到頁面
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // 調整輸出大小以符合視訊元素
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // 等待使用者點擊拍照按鈕
      await new Promise((resolve) => {
        capture.onclick = resolve;
      });

      // 將視訊幀繪製到Canvas上並轉換為JPEG格式
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop(); // 停止視訊串流
      div.remove(); // 移除介面元素
      return canvas.toDataURL('image/jpeg', quality); // 回傳照片的base64編碼
    }
    ''')
  display(js) # 顯示JavaScript介面
  # 執行JavaScript函數, 取得照片資料
  data = eval_js('takePhoto({})'.format(quality))
  # 解碼base64資料並寫入檔案
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename # 回傳儲存的檔案路徑

## setup_video_stream

In [9]:
# 設定JavaScript持續獲取鏡頭影像的函數
# 這個函數會建立一個介面,持續從使用者的攝影機取得影像
def setup_video_stream():
    """
    設定JavaScript持續獲取鏡頭影像的環境

    這個函數建立一個頁面介面,包含視訊顯示區域和狀態標籤,
    並設定回調函數來處理每一幀影像。
    """
    # 用於設定視訊串流處理環境
    js = Javascript('''
    var video; // 視訊元素
    var div = null; // 容器元素
    var stream; // 媒體串流
    var captureCanvas; // 用於攝取視訊幀的Canvas
    var imgElement; // 用於顯示處理後影像的圖片元素
    var labelElement; // 用於顯示狀態文字的元素

    var pendingResolve = null; // 用於處理非同步操作的Promise resolver
    var shutdown = false; // 標記是否關閉視訊串流

    // 移除DOM元素並清理資源的函數
    function removeDom() {
        stream.getVideoTracks()[0].stop(); // 停止視訊串流
        video.remove(); // 移除視訊元素
        div.remove(); // 移除容器
        // 重設所有變數
        video = null;
        div = null;
        stream = null;
        imgElement = null;
        captureCanvas = null;
        labelElement = null;
    }

    // 動畫幀回調函數,持續處理視訊
    function onAnimationFrame() {
        if (!shutdown) {
            window.requestAnimationFrame(onAnimationFrame); // 請求下一幀
        }
        if (pendingResolve) {
            var result = "";
            if (!shutdown) {
                // 將視訊幀繪製到Canvas上
                captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
                result = captureCanvas.toDataURL('image/jpeg', 0.8); // 將Canvas轉為JPEG
            }
            // 解析Promise並傳回結果
            var lp = pendingResolve;
            pendingResolve = null;
            lp(result);
        }
    }

    // 建立DOM元素並設定視訊串流的非同步函數
    async function createDom() {
        if (div !== null) {
            return stream; // 如果已經建立,直接回傳現有串流
        }

        // 建立容器元素
        div = document.createElement('div');
        div.style.border = '2px solid black';
        div.style.padding = '3px';
        div.style.width = '100%';
        div.style.maxWidth = '600px';
        div.style.margin = '0 auto';
        document.body.appendChild(div);

        // 建立狀態顯示區域
        const modelOut = document.createElement('div');
        modelOut.innerHTML = "<span>狀態: </span>";
        labelElement = document.createElement('span');
        labelElement.innerText = '準備中...';
        labelElement.style.fontWeight = 'bold';
        modelOut.appendChild(labelElement);
        div.appendChild(modelOut);

        // 建立視訊元素
        video = document.createElement('video');
        video.style.display = 'block';
        video.width = div.clientWidth - 6;
        video.setAttribute('playsinline', ''); // 允許在行內播放(iOS需要)
        video.onclick = () => { shutdown = true; }; // 點擊視訊停止程式
        // 請求使用者攝影機權限
        stream = await navigator.mediaDevices.getUserMedia(
            {video: { facingMode: "user"}}); // 設定使用前置攝影機
        div.appendChild(video);

        // 建立用於顯示處理後影像的圖片元素
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.onclick = () => { shutdown = true; }; // 點擊圖片停止程式
        div.appendChild(imgElement);

        // 建立使用說明元素
        const instruction = document.createElement('div');
        instruction.innerHTML =
            '<span style="color: red; font-weight: bold;">' +
            '*完成時,點擊影像區域停止濾鏡效果</span>';
        div.appendChild(instruction);
        instruction.onclick = () => { shutdown = true; }; // 點擊說明停止程式

        // 設定視訊源並開始播放
        video.srcObject = stream;
        await video.play();

        // 建立用於抓取視訊幀的Canvas
        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640; // 設定固定寬度
        captureCanvas.height = 480; // 設定固定高度
        // 啟動動畫幀處理
        window.requestAnimationFrame(onAnimationFrame);

        return stream;
    }
    // 處理每一幀視訊和顯示處理後影像的非同步函數
    async function stream_frame(label, imgData) {
      // 如果標記為關閉, 則清理DOM並退出
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom(); // 確保DOM已建立

      var preShow = Date.now();
      // 更新狀態標籤
      if (label != "") {
        labelElement.innerHTML = label;
      }

      // 如果有處理後的影像資料, 則顯示
      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        // 設定圖片元素的位置和大小與視訊相同
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData; // 設定圖片源
      }

      var preCapture = Date.now();
      // 等待下一幀的處理結果
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;

      // 回傳處理時間統計和結果影像
      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

    display(js) # 顯示並執行上述JavaScript程式碼

## video_frame

In [10]:
# 處理單一視訊幀的函數
def video_frame(label, bbox):
    """
    處理單一視訊幀

    參數:
        label (str): 要顯示的狀態文字
        bbox (str): 處理後的影像資料, Base64格式

    回傳:
        dict: 包含處理時間和影像資料的字典
    """
    # 執行JavaScript函數處理當前視訊幀
    data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
    return data

## js_to_image

In [11]:
# 將JavaScript輸出的影像資料轉換為OpenCV影像格式
def js_to_image(js_reply):
    """
    將JavaScript輸出的Base64編碼影像轉換為OpenCV影像格式

    參數:
        js_reply (str): JavaScript回傳的Base64編碼影像資料

    回傳:
        numpy.ndarray: OpenCV格式的影像
    """
    # 解碼Base64資料
    image_bytes = b64decode(js_reply.split(',')[1])
    # 使用PIL開啟影像
    image = PILImage.open(io.BytesIO(image_bytes))
    # 轉換為NumPy陣列
    image = np.array(image)
    # 轉換色彩空間從RGB(PIL)到BGR(OpenCV)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image

## SmileFilterApp

In [16]:
# 微笑濾鏡應用類別
class SmileFilterApp:
    """
    微笑濾鏡應用類別

    此類別使用MediaPipe的FaceMesh功能檢測臉部特徵點,
    並根據特定的特徵點判斷使用者是否在微笑,
    當偵測到微笑時, 會套用特殊的濾鏡效果。
    """

    def __init__(self):
        """初始化微笑濾鏡應用"""
        # 初始化 MediaPipe Face Mesh 用於臉部特徵點檢測
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            max_num_faces=1,   # 只檢測一張臉
            min_detection_confidence=0.5,  # 檢測信心閾值
            min_tracking_confidence=0.5)   # 追蹤信心閾值

        # 初始化 MediaPipe Drawing 用於繪製特徵點和連線
        self.mp_drawing = mp.solutions.drawing_utils
        # self.drawing_spec = self.mp_drawing.DrawingSpec(thickness=1, circle_radius=1) # 設定繪製樣式
        self.landmark_spec = self.mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=1, circle_radius=1) # 點改紅色
        self.connection_spec = self.mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1) # 線改綠色

        # 設定濾鏡狀態相關變數
        self.filter_active = False  # 濾鏡是否啟用
        self.smile_counter = 0      # 計數檢測到微笑的連續幀數
        self.smile_threshold = 3    # 啟用濾鏡需要連續檢測到微笑的幀數閾值
        self.smile_detected_flag = False # 是否曾經檢測到微笑的標記

        # 微笑最小值設定 - 濾鏡會在低於此值時自動取消
        self.min_smile_intensity = 0.6 # 微笑程度必須達到60%才會保持保持濾鏡效果

        # 定義關鍵臉部特徵點索引
        # MediaPipe FaceMesh提供468個面部標記點, 以下是嘴巴相關的重要標記點
        self.LEFT_MOUTH_CORNER = 61      # 左嘴角的特徵點索引
        self.RIGHT_MOUTH_CORNER = 291    # 右嘴角的特徵點索引
        self.TOP_LIP = 13                # 上唇中間的特徵點索引
        self.BOTTOM_LIP = 14             # 下唇中間的特徵點索引

        self.prev_intensity = 0.0

    def detect_smile(self, face_landmarks):
      """
      檢測是否微笑

      通過分析嘴巴的形狀和特徵點位置關係來判斷是否為微笑

      參數:
          face_landmarks: MediaPipe檢測到的臉部特徵點

      回傳:
          tuple: (是否微笑的布林值, 微笑強度的浮點數)
      """
      # 獲取嘴角和嘴唇的特徵點座標
      left_mouth = face_landmarks.landmark[self.LEFT_MOUTH_CORNER]
      right_mouth = face_landmarks.landmark[self.RIGHT_MOUTH_CORNER]
      top_lip = face_landmarks.landmark[self.TOP_LIP]
      bottom_lip = face_landmarks.landmark[self.BOTTOM_LIP]

      # 計算嘴巴的寬度和高度
      mouth_width = abs(right_mouth.x - left_mouth.x) # 嘴巴寬度
      mouth_height = abs(top_lip.y - bottom_lip.y)    # 嘴巴高度

      # 計算嘴巴寬高比 - 微笑時嘴巴會變寬而高度減少
      mouth_ratio = mouth_width / (mouth_height + 1e-5) # 加上極小值避免除以零

      # 計算嘴角的高度比例 - 檢查嘴角是否確實上揚
      left_corner_height = (left_mouth.y - top_lip.y) # 左嘴角相對於上唇的位置
      right_corner_height = (right_mouth.y - top_lip.y) # 右嘴角相對於上唇的位置
      corner_avg = (left_corner_height + right_corner_height) / 2 # 兩嘴角高度的平均值

      # 檢查中間嘴唇是否低於嘴角 (真正的微笑特徵)
      middle_lip_y = (top_lip.y + bottom_lip.y) / 2 # 嘴唇中間點的垂直位置
      mouth_curve = (left_mouth.y + right_mouth.y) / 2 - middle_lip_y # 嘴角與中間點的垂直差距

      smile_threshold_ratio = 4.0        # 嘴巴寬高比閾值
      smile_corner_threshold = 0.03      # 嘴角高度閾值

      # 要求嘴巴寬高比大於閾值、嘴角位置較高, 以及嘴角確實上揚
      is_smiling = (mouth_ratio > smile_threshold_ratio and
                    corner_avg < smile_corner_threshold and
                    mouth_curve < -0.01) # 確保嘴巴呈微笑弧度

      # 計算微笑強度 - 用於調整濾鏡效果的強度
      # 將寬高比映射到0到1.0的強度值範圍
      smile_intensity = min(max((mouth_ratio-smile_threshold_ratio) / smile_threshold_ratio, 0.0), 1.0) # 限制最大值

      return is_smiling, smile_intensity

    def apply_oilpainting_filter(self, image, intensity=1.0):
      """
      將影像轉換為油畫風格
      利用中值濾波處理來達到筆觸效果

      參數:
          image (numpy.ndarray): 輸入的OpenCV影像
          intensity (float): 效果強度, 預設為1.0
      回傳:
          numpy.ndarray: 套用了油畫效果的影像
      """

      # 強度越高, 影像越模糊, 筆觸感越重
      brush_size = int(intensity * 5)
      if brush_size % 2 == 0: brush_size += 1
      brush_size = max(3, brush_size)

      # 中值濾波減少雜訊
      img_blur = cv2.medianBlur(image, brush_size)

      # 使用雙邊濾波保留臉部特徵
      oil_effect = cv2.bilateralFilter(img_blur, d=9, sigmaColor=75, sigmaSpace=75)

      # 增加色彩飽和度
      hsv = cv2.cvtColor(oil_effect, cv2.COLOR_BGR2HSV)
      hsv[:, :, 1] = np.minimum(hsv[:, :, 1] * 1.2, 255)

      return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # 素描
    def apply_sketch_filter(self, image, intensity=1.0):
        # 產生素描圖
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        inv = cv2.bitwise_not(gray)
        blur = cv2.GaussianBlur(inv, (21, 21), 0)
        sketch = cv2.divide(gray, 255 - blur, scale=256)
        sketch_bgr = cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)

        # 2. 將 intensity 限制在 0.0 到 1.0 之間
        alpha = min(max(intensity, 0.0), 1.0)

        # 影像融合 intensity 比例的素描圖 + (1-intensity) 比例的原圖
        # 微笑越用力，素描感越深
        blended = cv2.addWeighted(sketch_bgr, alpha, image, 1 - alpha, 0)

        return blended

    # 可以替換成其他濾鏡效果
    def apply_rainbow_filter(self, image, intensity=1.0):
      """
      應用彩虹濾鏡效果

      將影像轉換為HSV色彩空間, 然後按照水平位置改變色相,
      創造出彩虹般的漸變效果

      參數:
          image (numpy.ndarray): 輸入的OpenCV影像
          intensity (float): 效果強度, 預設為1.0

      回傳:
          numpy.ndarray: 套用了彩虹效果的影像
      """
      # 將BGR影像轉換為HSV色彩空間
      hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

      # 獲取影像尺寸
      height, width = image.shape[:2]

      # 使用向量化操作替代循環以提高速度
      # 根據x座標計算色相偏移量
      x_coords = np.arange(width) # 建立從0到width-1的陣列
      hue_shifts = (x_coords / width * 180 * intensity) % 180 # 將x座標映射到0-180的色相範圍

      # 為每一行應用漸變色相效果
      for i in range(height):
          hsv[i, :, 0] = hue_shifts # 設定色相通道
          hsv[i, :, 1] = np.minimum(hsv[i, :, 1] * 1.5, 255) # 增加飽和度

      # 將HSV影像轉回BGR色彩空間
      return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    def draw_smile_meter(self, image, intensity):
        h, w = image.shape[:2]
        # 進度條位置與大小
        bar_x = 30
        bar_y = int(h * 0.2)
        bar_w = 20
        bar_h = int(h * 0.6)

        cv2.rectangle(image, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (50, 50, 50), -1)
        cv2.rectangle(image, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (255, 255, 255), 2)

        fill_h = int(bar_h * intensity)
        fill_y = bar_y + bar_h - fill_h

        # 進度條
        if fill_h > 0:
            cv2.rectangle(image, (bar_x, fill_y), (bar_x + bar_w, bar_y + bar_h), (0, 255, 0), -1)

        # 顯示強度數值文字 (小數點後兩位)
        text = f"Intensity: {intensity:.2f}"
        cv2.putText(image, text, (bar_x, bar_y - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        return image

    def process_frame(self, frame):
      """
      處理單幀影像

      檢測臉部、判斷微笑, 並根據檢測結果套用濾鏡效果

      參數:
          frame (numpy.ndarray): 輸入的視訊幀

      回傳:
          numpy.ndarray: 處理後的視訊幀
      """
      # 水平翻轉圖片, 使其像鏡子一樣
      image = cv2.flip(frame, 1)

      # 轉RGB色彩空間供MediaPipe處理
      # MediaPipe需要RGB格式, 而OpenCV預設使用BGR
      image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

      # 使用MediaPipe處理影像, 檢測臉部特徵點
      results = self.face_mesh.process(image_rgb)

      # 取得圖像尺寸
      h, w = image.shape[:2]

      # 初始化微笑強度
      current_smile_intensity = 0.0

      # 如果檢測到臉部
      if results.multi_face_landmarks:
          for face_landmarks in results.multi_face_landmarks:
              # 檢測是否微笑
              is_smiling, smile_intensity = self.detect_smile(face_landmarks)
              # current_smile_intensity = smile_intensity # 儲存當前微笑強度
              target_intensity = smile_intensity if is_smiling else 0.0
              self.prev_intensity = (0.2 * target_intensity) + (0.8 * self.prev_intensity)
              current_smile_intensity = self.prev_intensity

              # 更新微笑計數器
              if is_smiling:
                  # 上限設定在 3 幀 (避免已經停止微笑但笑太久會太久才關掉濾鏡)
                  self.smile_counter = min(self.smile_counter + 1, self.smile_threshold + 3) # 連續檢測到微笑的幀數增加
                  if self.smile_counter > self.smile_threshold:
                      self.smile_detected_flag = True # 標記已經偵測到笑容
              else:
                  # 如果當前幀沒有檢測到微笑, 計數器降低
                  self.smile_counter = max(0, self.smile_counter - 1)

              # 決定是否啟用濾鏡
              # 需要連續多幀檢測到微笑才啟用濾鏡
              if self.smile_counter > self.smile_threshold:
                  self.filter_active = True
              elif self.smile_counter < 2: # 如果連續多幀未檢測到微笑則停用濾鏡
                  self.filter_active = False

              # 如果微笑程度低於閾值, 則關閉濾鏡
              # 即使計數器夠高, 但微笑不夠明顯也不套用濾鏡
              if current_smile_intensity < self.min_smile_intensity:
                  self.filter_active = False

              # 顯示微笑程度 (此功能已被註解)
              image = self.draw_smile_meter(image, current_smile_intensity)

              # 根據濾鏡狀態應用效果
              if self.filter_active:
                  # 對高強度微笑使用彩虹效果
                  # 只有當微笑強度超過0.9時才套用彩虹濾鏡
                  # if smile_intensity > 0.6:
                  #    image = self.apply_rainbow_filter(image, intensity=smile_intensity)
                  # if smile_intensity > 0.9:
                      # image = self.apply_hybrid_filter(image, intensity=smile_intensity)
                  image = self.apply_sketch_filter(image, intensity=current_smile_intensity)


              # 顯示文字提示 (此功能已被註解)
              # cv2.putText(image, "已偵測到微笑!", (w // 2 - 120, 50),
              #             cv2.FONT_HERSHEY_DUPLEX, 1, (0, 255, 0), 2)

              # 繪製臉部網格 - 顯示MediaPipe檢測到的所有臉部特徵點
              self.mp_drawing.draw_landmarks(
                  image=image,
                  landmark_list=face_landmarks,
                  connections=self.mp_face_mesh.FACEMESH_TESSELATION, # 使用TESSELATION連線模式
                  landmark_drawing_spec=self.landmark_spec, # 設定特徵點繪製樣式
                  connection_drawing_spec=self.connection_spec) # 設定連線繪製樣式


      # 回傳處理後的影像
      return image

    def run_on_colab(self, num_frames=200):
      """
      在Colab環境中運行微笑濾鏡應用

      設定視訊串流, 處理每一幀, 並顯示結果

      參數:
          num_frames (int): 要處理的最大幀數, 預設為200
      """
      # 設置鏡頭和視訊處理環境
      setup_video_stream()

      try:
          # 運行指定的幀數或直到手動停止
          for i in range(num_frames):
              # 建立狀態標籤, 根據當前檢測狀態顯示不同訊息
              if self.smile_detected_flag and self.filter_active:
                  status = "已偵測到微笑！正在應用濾鏡效果"
              elif self.smile_detected_flag and not self.filter_active:
                  status = "微笑程度不足，請加大微笑！"
              else:
                  status = f"請對著攝像頭微笑 (幀 {i+1}/{num_frames})"

              # 獲取當前視訊幀
              frame_data = video_frame(status, "")

              # 如果使用者關閉了視訊, 結束處理
              if frame_data['img'] == '':
                  print("結束了影像使用")
                  break

              # 處理圖像 - 將JS回傳的資料轉換為OpenCV格式, 然後處理
              frame = js_to_image(frame_data['img'])
              processed_frame = self.process_frame(frame)

              # 將處理後的幀轉換為base64以顯示
              # 先編碼為JPEG, 再轉為base64字串
              _, buffer = cv2.imencode('.jpg', processed_frame)
              img_str = b64encode(buffer).decode('utf-8')
              # _, buffer = cv2.imencode('.jpg', processed_frame)
              # img_str = b64encode(buffer).decode('utf-8')

              # 顯示處理後的結果, 並更新狀態標籤
              frame_data = video_frame(status, f"data:image/jpeg;base64,{img_str}")

              # 如果用戶停止則退出
              if frame_data['img'] == '':
                  print("停止了")
                  break

              # 適度暫停以減輕計算負擔, 但不要太長
              # 這可以讓UI更有回應性, 並降低CPU使用率
              time.sleep(0.05)

      except Exception as e:
          # 捕獲並輸出錯誤
          print(f"發生錯誤: {e}")

# 執行

In [21]:
smile_app = SmileFilterApp()

smile_app.run_on_colab(num_frames=200)

<IPython.core.display.Javascript object>

發生錯誤: string indices must be integers, not 'str'
